# Qwen3.5-0.8B — inside the model

This is the **analysis workshop**. The everyday C++ chat runs separately with `tools/start-chat.sh` at http://127.0.0.1:8088.

**Start here:** select **Run → Run All Cells**. Then press **Analyze prompt** below. Running this notebook only creates the controls; it does not load model weights until you request an experiment.

The model is the agreed **Qwen3.5-0.8B**, using its original BF16 checkpoint. The C++ chat uses Q4_K_M weights of the same model, so output probabilities can differ. CPU inference uses two threads. Keep one model runtime loaded at a time.


In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))
from playground import dashboard
lab = dashboard()


## 1. Tokens and the next-token distribution

Start with `The capital of Hungary is` in **raw text** mode. Analyze, then open **Tokens** and **Next token**. A token can be a word fragment, punctuation or a control symbol. Individual decoded fragments may not form a complete Unicode character.

The table shows the ten largest probabilities after softmax over the **entire vocabulary**. These are next-token probabilities, not the probability that an answer is true. Try `The capital of France is` and compare.

Enable **Chat template** to see the role delimiters and the non-thinking assistant prefix that the model actually receives. **Generate reply** always uses this chat template, with thinking disabled. Generation is capped at 64 new tokens; a long answer can therefore be cut off.

## 2. Attention — six layers, eight heads each

In **Attention map**, choose actual model layer **3, 7, 11, 15, 19 or 23** and head **0–7**. The horizontal axis is the token being attended to; the vertical axis is the query token. The future-token triangle is masked.

**BertViz** shows the same captured weights as interactive connections. Its compact layer selector **0–5** maps to model layers **3, 7, 11, 15, 19, 23**. It does not visualize all 24 layers as ordinary attention.

Qwen3.5 is a hybrid: **three Gated DeltaNet blocks, then one full-attention block**, repeated six times. DeltaNet does not expose the same softmax token-to-token attention matrix. An attention line is not a proven causal explanation.

## 3. Activations — all 24 decoder layers

**All 24 layers** shows the root-mean-square (RMS) magnitude of each decoder block's output, over its 1024 hidden dimensions, at every token position. This includes DeltaNet and full-attention blocks, after their feed-forward/residual computation.

Brightness measures magnitude, **not importance, confidence or a named concept**. The plot does not decode DeltaNet recurrent memory or identify individual semantic features. BertViz's specialized neuron view is not enabled for this architecture.

## 4. A controlled intervention

Select **MLP layer 0**, then **Compare MLP ablation**. This runs the same prompt twice: once normally, once replacing one block's feed-forward output with zero. It compares next-token distributions.

Total variation is between 0 (identical distributions) and 1 (disjoint distributions). A small value is still a valid result; changing a component need not change the top token. The hook is removed even if inference fails. Model weights are never edited, and this experiment is not training or permanent learning.

## 5. Resource limits and shutdown

- At most **96 input tokens**, including chat control tokens; overlong input is rejected rather than silently truncated.
- At most **64 generated tokens**. Two CPU threads, one experiment at a time.
- **Unload model** releases the model reference. **Kernel → Shut Down Kernel** fully ends the Python process and releases its memory.
- Closing the browser tab alone does not stop Jupyter or a loaded kernel.
- Shut down Jupyter with **File → Shut Down**, or Ctrl+C in its terminal. Stop llama.cpp with Ctrl+C in its own terminal.
- No systemd unit, cron task, desktop autostart item, login hook or Docker restart policy is installed.

See [the full guide](../docs/QWEN_PLAYGROUND_GUIDE.md) and the PDF in `docs/`.
